> 在线讲义：[第 4 章 · 量化：省内存、省带宽，但不一定变快](https://wikieden.github.io/robot-edge-inference/lessons/04-quantization.html) · 本书《AI Infra 扩展 · 机器人端侧推理》

# 第 4 章 · 量化：省内存、省带宽，但不一定变快

配套讲义：仓库里 `ai-infra-edge/notes/lessons/04-quantization.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import math
import random

print("纯 Python，不需要安装任何库")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（OpenVLA 的数来自论文 Table 2；5 个权重是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 7B 各格式大小 | 参数 × 字节 | `weights_gb(7, "int4")` | 3.50 GB（bf16 14 GB） |
| 第一步 q4_0 一组 | $16 + 2$ | `q4_0_block_bytes()` | 18 字节 |
| 第二步 int8 量化 | $\max|x| / 127$ | `quantize(xs, 8)` | 整数 17、−71、47、127、−10 |
| 第二步 int4 量化 | $\max|x| / 7$ | `quantize(xs, 4)` | 整数 1、−4、3、7、−1 |
| 第二步 int4 还原误差 | $\max|x - \hat x|$ | `dequantize(...)` | 0.0586 |

OpenVLA 的量化表来自论文 Table 2；q4_0 的字节数来自 AII §8.4；那 5 个权重是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
BYTES_PER_VALUE = {"bf16": 2.0, "fp8": 1.0, "int8": 1.0, "int4": 0.5, "q4_0": 18 / 32}
OPENVLA = {"bf16": (71.3, 16.8), "int8": (58.1, 10.2), "int4": (71.9, 7.0)}  # (成功率 %, 显存 GB)

**函数 `weights_gb`**：权重占多少 GB：参数量 × 每个值的字节数。

In [ ]:
def weights_gb(params_billion: float, fmt: str) -> float:
    """权重占多少 GB：参数量 × 每个值的字节数。"""
    return params_billion * BYTES_PER_VALUE[fmt]

**函数 `q4_0_block_bytes`**：q4_0 一组 32 个值：每个值 4 位（共 16 字节）+ 一个 2 字节的缩放系数。

In [ ]:
def q4_0_block_bytes(n_values: int = 32) -> int:
    """q4_0 一组 32 个值：每个值 4 位（共 16 字节）+ 一个 2 字节的缩放系数。"""
    return n_values * 4 // 8 + 2

**函数 `quantize`**：对称量化：scale = 最大绝对值 ÷ 最大整数，每个值除以 scale 再四舍五入。

In [ ]:
def quantize(xs: list[float], bits: int) -> tuple[float, list[int]]:
    """对称量化：scale = 最大绝对值 ÷ 最大整数，每个值除以 scale 再四舍五入。"""
    qmax = 2 ** (bits - 1) - 1
    scale = max(abs(x) for x in xs) / qmax
    return scale, [round(x / scale) for x in xs]

**函数 `dequantize`**：反量化：整数乘回 scale。

In [ ]:
def dequantize(scale: float, qs: list[int]) -> list[float]:
    """反量化：整数乘回 scale。"""
    return [q * scale for q in qs]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：7B 模型在不同格式下的权重大小")
for fmt in BYTES_PER_VALUE:
    print(f"  {fmt:<5} {BYTES_PER_VALUE[fmt]:.4g} 字节 / 值 → {weights_gb(7, fmt):5.2f} GB")
print(f"  q4_0 一组 32 个值 = {q4_0_block_bytes()} 字节（bf16 要 64 字节），平均 {q4_0_block_bytes() * 8 / 32} 位")

In [ ]:
print("\n第二步：亲手量化 5 个权重（编的）")
xs = [0.12, -0.5, 0.33, 0.9, -0.07]
for bits in (8, 4):
    scale, qs = quantize(xs, bits)
    back = dequantize(scale, qs)
    err = max(abs(a - b) for a, b in zip(xs, back))
    print(f"  int{bits}：scale = 0.9 ÷ {2 ** (bits - 1) - 1} = {scale:.5f}，整数 {qs}")
    print(f"         还原 {[round(b, 3) for b in back]}，最大误差 {err:.4f}")

In [ ]:
print("\n第三步：OpenVLA 的真实结果（论文 Table 2）")
for fmt, (succ, mem) in OPENVLA.items():
    print(f"  {fmt:<5} 成功率 {succ}%  显存 {mem} GB")
print("  int8 更省内存却更差：多出的量化算子让推理变慢，A5000 上只有 1.2 Hz")

## 3. 代码里才看得到的

- Python 的 `round` 是"四舍六入五成双"（0.5 舍到偶数），和有些硬件的舍入规则不同；本章例子里没碰到正好 .5 的情况。
- 真实框架里 scale 也有精度（q4_0 用 2 字节的 fp16 存），这里用的是 Python 的双精度浮点，误差会略小一点。

## 4. 自己改一改

- 把 `xs` 里的 0.9 改成 3.0（加一个离群值），看 int4 的误差变成多少——这就是要分组的原因。
- 把 `quantize` 改成两组各算一个 scale（前 3 个一组、后 2 个一组），看误差下降多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 位数减半字节减半，带宽内存一起省；
> 最大绝对除最大整，四舍五入对格子；
> 只压权重省带宽，连激活压才省算；
> 变慢一样会做坏，数值仿真加真机。